## Clean Placeholder Author Names

DataCite metadata sometimes carries CDL/EZID **Kernel Metadata Reserved Codes** — sentinel values used when a `creators[].name` field is unavailable:

| Code | Meaning |
|---|---|
| `(:unav)` | value unavailable |
| `(:unkn)` | known to be unknown |
| `(:unap)` | not applicable |
| `(:unas)` | unassigned |
| `(:unal)` | unallowed/suppressed |
| `(:unac)` | temporarily inaccessible |

Some appear with literal trailing text (`(:Unkn) Unknown`, `(:Unas) Unassigned`, `(:unav) Unknown author`).

These strings flow from `openalex.datacite.datacite_parsed.authors[].name` through `openalex_works_base` into `openalex.works.work_authors.raw_author_name`. The name parser turns them into nonsense `parsed_name.first/middle/last` values, which corrupts `block_key` derivation and the `name_variants` set in `authors_for_matching`. Author profiles whose `full_name` is itself a placeholder (e.g. A5108365034 with display_name=`(:Unav)`) are matching black-holes that aggregate hundreds of unrelated works.

**Scope** at time of authoring:
- ~778K rows in `work_authors` carry a placeholder string
- 4,103 distinct author profiles touched
- 3,833 are *mixed* profiles (some real names, some placeholders) — safe to clean by nulling the placeholder rows
- 324 profiles have `full_name` that is itself a placeholder (1 has ORCID, the rest are anonymous)
- 282 pure-placeholder profiles will end up with empty `raw_author_names` after CreateAuthors rebuilds; flagged for follow-up purge

**Forward fix is separate**: a regex check should be added to `notebooks/ingest/DataCite.py` (~line 160, `attributes.creators` transform) and to `clean_author_name` in `libraries/dlt_utils/openalex/dlt/normalize.py` so future ingestions never let these through. This notebook only cleans the warehouse state.

**Regex used**: `^\s*\(:un[a-z]{2,3}\)(\s*(unknown( author)?|unassigned))?\s*$` (case-insensitive).

**Downstream rebuild**: after the UPDATEs in cells 8–9, run `notebooks/authors/CreateAuthors.ipynb` so `openalex_authors.raw_author_names` and `authors_for_matching.name_variants` rebuild without the placeholders. The aggregation already uses `ARRAY_DISTINCT(ARRAY_COMPACT(...))`, which drops NULLs.

### Cell 1: Build candidate rows in work_authors to null

Snapshot every `(work_id, author_sequence)` pair where `raw_author_name` matches the placeholder regex. Includes the matched `author_id` (may be NULL — those rows are inert but still get nulled for tidiness).

In [ ]:
CREATE OR REPLACE TABLE openalex.authors.placeholder_name_cleanup_candidates AS
SELECT
  work_id,
  author_sequence,
  author_id,
  raw_author_name AS previous_raw_author_name
FROM openalex.works.work_authors
WHERE raw_author_name IS NOT NULL
  AND LOWER(raw_author_name) RLIKE '^\\s*\\(:un[a-z]{2,3}\\)(\\s*(unknown( author)?|unassigned))?\\s*$'

### Cell 2: Validation — variant breakdown and distinct authors touched

In [ ]:
SELECT
  COUNT(*) AS total_rows_to_null,
  COUNT(DISTINCT author_id) AS distinct_authors_touched,
  COUNT(DISTINCT work_id) AS distinct_works_touched,
  SUM(CASE WHEN author_id IS NULL THEN 1 ELSE 0 END) AS rows_with_null_author_id
FROM openalex.authors.placeholder_name_cleanup_candidates

### Cell 3: Variant breakdown spot-check

In [ ]:
SELECT previous_raw_author_name, COUNT(*) AS n,
       COUNT(DISTINCT author_id) AS distinct_authors
FROM openalex.authors.placeholder_name_cleanup_candidates
GROUP BY previous_raw_author_name
ORDER BY n DESC

### Cell 4: Profile candidates — authors with placeholder full_name

These profiles have `full_name` set to a placeholder string. Most are anonymous (no ORCID); we will null `full_name` so the matcher stops keying off the placeholder. `display_name` is left untouched here — pure-placeholder profiles get flagged in Cell 7 for follow-up review.

In [ ]:
CREATE OR REPLACE TABLE openalex.authors.placeholder_profile_cleanup_candidates AS
SELECT
  id AS author_id,
  display_name AS current_display_name,
  full_name AS previous_full_name,
  orcid
FROM openalex.authors.authors
WHERE full_name IS NOT NULL
  AND LOWER(full_name) RLIKE '^\\s*\\(:un[a-z]{2,3}\\)(\\s*(unknown( author)?|unassigned))?\\s*$'

### Cell 5: Profile candidates validation

In [ ]:
SELECT
  COUNT(*) AS total_profiles,
  SUM(CASE WHEN orcid IS NOT NULL AND orcid <> '' THEN 1 ELSE 0 END) AS profiles_with_orcid,
  COUNT(DISTINCT previous_full_name) AS distinct_placeholder_fullnames
FROM openalex.authors.placeholder_profile_cleanup_candidates

### Cell 6: Pure-placeholder profile audit (informational only — NOT modified by this notebook)

Profiles whose every `raw_author_names[]` entry is a placeholder. After Cell 8 nulls all the placeholder rows in `work_authors` and CreateAuthors rebuilds, these profiles will end up with empty `raw_author_names` arrays. They are matching black-holes (e.g. A5133762328 has 1509 works all attached via the `(:unav)` placeholder). Recommend a separate notebook to delete these profiles and null their `work_authors.author_id` references.

In [ ]:
CREATE OR REPLACE TABLE openalex.authors.placeholder_profile_purge_audit AS
WITH per_author AS (
  SELECT
    oa.id, oa.display_name, oa.full_name, oa.orcid, oa.works_count,
    oa.raw_author_names,
    SIZE(oa.raw_author_names) AS total_names,
    SIZE(FILTER(oa.raw_author_names,
                n -> LOWER(n) RLIKE '^\\s*\\(:un[a-z]{2,3}\\)(\\s*(unknown( author)?|unassigned))?\\s*$'))
      AS placeholder_names
  FROM openalex.authors.openalex_authors oa
)
SELECT id AS author_id, display_name, full_name, orcid, works_count, raw_author_names
FROM per_author
WHERE total_names > 0 AND total_names = placeholder_names

### Cell 7: Create combined audit log (for rollback)

In [ ]:
CREATE OR REPLACE TABLE openalex.authors.placeholder_name_cleanup_log AS
SELECT 'work_author_row' AS scope,
  CAST(work_id AS STRING) AS work_id,
  CAST(author_sequence AS STRING) AS author_sequence,
  CAST(author_id AS STRING) AS author_id,
  previous_raw_author_name AS previous_value,
  CAST(NULL AS STRING) AS new_value,
  current_timestamp() AS cleanup_timestamp
FROM openalex.authors.placeholder_name_cleanup_candidates
UNION ALL
SELECT 'author_full_name' AS scope,
  CAST(NULL AS STRING) AS work_id,
  CAST(NULL AS STRING) AS author_sequence,
  CAST(author_id AS STRING) AS author_id,
  previous_full_name AS previous_value,
  CAST(NULL AS STRING) AS new_value,
  current_timestamp() AS cleanup_timestamp
FROM openalex.authors.placeholder_profile_cleanup_candidates

### Cell 8: Execute work_authors cleanup

**WARNING**: Updates `openalex.works.work_authors`. Verify cells 2–3 before running. The `MERGE` is gated on the previous value matching the snapshot to avoid clobbering anything that changed since cell 1.

In [ ]:
MERGE INTO openalex.works.work_authors AS target
USING openalex.authors.placeholder_name_cleanup_candidates AS source
  ON target.work_id = source.work_id
     AND target.author_sequence = source.author_sequence
WHEN MATCHED AND target.raw_author_name = source.previous_raw_author_name THEN
  UPDATE SET
    target.raw_author_name = NULL,
    target.updated_at = current_timestamp()

### Cell 9: Execute authors.full_name cleanup

**WARNING**: Updates `openalex.authors.authors`. The 1 profile with an ORCID retains its `display_name` and `orcid` — only `full_name` is nulled, so it falls out of `block_key` derivation but the identity is preserved.

In [ ]:
MERGE INTO openalex.authors.authors AS target
USING openalex.authors.placeholder_profile_cleanup_candidates AS source
  ON target.id = source.author_id
WHEN MATCHED AND target.full_name = source.previous_full_name THEN
  UPDATE SET
    target.full_name = NULL,
    target.updated_date = current_timestamp()

### Cell 10: Post-cleanup verification

In [ ]:
SELECT 'work_authors_remaining' AS check_name,
       COUNT(*) AS remaining_placeholders
FROM openalex.works.work_authors
WHERE raw_author_name IS NOT NULL
  AND LOWER(raw_author_name) RLIKE '^\\s*\\(:un[a-z]{2,3}\\)(\\s*(unknown( author)?|unassigned))?\\s*$'
UNION ALL
SELECT 'authors_full_name_remaining' AS check_name,
       COUNT(*) AS remaining_placeholders
FROM openalex.authors.authors
WHERE full_name IS NOT NULL
  AND LOWER(full_name) RLIKE '^\\s*\\(:un[a-z]{2,3}\\)(\\s*(unknown( author)?|unassigned))?\\s*$'
UNION ALL
SELECT 'log_rows' AS check_name, COUNT(*) AS remaining_placeholders
FROM openalex.authors.placeholder_name_cleanup_log

### Cell 11: Rollback template (use only if a systematic FP is discovered)

```sql
-- Restore work_authors raw_author_name from the log
MERGE INTO openalex.works.work_authors AS target
USING (SELECT CAST(work_id AS BIGINT) AS work_id,
              CAST(author_sequence AS BIGINT) AS author_sequence,
              previous_value
       FROM openalex.authors.placeholder_name_cleanup_log
       WHERE scope = 'work_author_row') AS source
  ON target.work_id = source.work_id
     AND target.author_sequence = source.author_sequence
WHEN MATCHED AND target.raw_author_name IS NULL THEN
  UPDATE SET target.raw_author_name = source.previous_value;

-- Restore authors.full_name from the log
MERGE INTO openalex.authors.authors AS target
USING (SELECT CAST(author_id AS BIGINT) AS author_id, previous_value
       FROM openalex.authors.placeholder_name_cleanup_log
       WHERE scope = 'author_full_name') AS source
  ON target.id = source.author_id
WHEN MATCHED AND target.full_name IS NULL THEN
  UPDATE SET target.full_name = source.previous_value;
```

Add a `WHERE` clause inside the USING subquery to restrict rollback to a specific class.

### Downstream rebuild

After cells 8–9, kick off `notebooks/authors/CreateAuthors.ipynb` so:
- `openalex.authors.openalex_authors.raw_author_names` drops the nulled placeholders (existing `ARRAY_DISTINCT(ARRAY_COMPACT(...))` handles it)
- `block_key` re-derives off the (now-nulled) `full_name` for the 324 profiles → they will have NULL `block_key` and drop out of `authors_for_matching`
- ES sync sees `updated_date` bumps on the 324 profiles plus all touched works (via the work-side rebuild)

`work_authorships` (the ES-sync nested form) rebuilds from `work_authors`; whichever job builds it should be re-run to propagate the cleanup downstream of the warehouse.

### Follow-up

`openalex.authors.placeholder_profile_purge_audit` lists the ~282 pure-placeholder profiles. Recommend a separate notebook to:
1. Set `author_id = NULL` on `work_authors` rows referencing these IDs
2. Delete these profile rows from `openalex.authors.authors`

Not done here because it has higher blast radius (changes `author_id` on hundreds of thousands of work_authors rows).